# 02 — Embedding Experiments: MiniLM vs Biomedical Models

Goal: understand the embedding model choices before building the FAISS index.

Questions:
1. What is the max sequence length of `all-MiniLM-L6-v2`? How much of our chunks get truncated?
2. How fast is embedding on 10K trials on Apple Silicon (MPS)?
3. What does retrieval quality look like? Can we construct a small labeled set to measure Precision@5 and MRR?
4. Where does MiniLM visibly fail — what query types hurt it most?
5. What would BioBERT/BioLinkBERT add? (planned for Component 4 upgrade)

**Key finding up front:** MiniLM max seq length = 256 tokens. Median chunk length = ~1862 chars.  
89% of our chunks exceed 1000 chars and are being severely truncated.  
The bi-encoder is essentially doing title + phase + conditions + first ~100 words of eligibility.  
This is the primary motivation for cross-encoder reranking in Component 4 — the reranker can see the full eligibility text.

In [ ]:
import numpy as np
import faiss
import sqlite_utils
import time
from pathlib import Path
from sentence_transformers import SentenceTransformer

DB_PATH = Path("../data/trialcompass.db")
db = sqlite_utils.Database(DB_PATH)

model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model: all-MiniLM-L6-v2")
print(f"Max seq length: {model.max_seq_length} tokens")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

## 1. Truncation analysis — how much are we losing?

In [ ]:
rows = list(db["trials"].rows_where(limit=500))
chunks = [r["chunk_text"] for r in rows]

# Rough token estimate: ~4 chars per token
char_limit = model.max_seq_length * 4  # ~1024 chars
lengths = [len(c) for c in chunks]

truncated = sum(1 for l in lengths if l > char_limit)
print(f"Chunks exceeding ~{char_limit} char limit: {truncated}/{len(chunks)} ({truncated/len(chunks)*100:.1f}%)")
print(f"Median chunk length: {sorted(lengths)[len(lengths)//2]:,} chars")
print(f"Max chunk length: {max(lengths):,} chars")
print(f"Mean chunk length: {sum(lengths)/len(lengths):,.0f} chars")
print()
print("What gets truncated:")
print("  The eligibility criteria text is typically last in the chunk.")
print("  For long trials, the model sees: title + phase + conditions + sponsor + first ~100 words of eligibility.")
print("  Inclusion/exclusion criteria beyond word ~100 are invisible to the bi-encoder.")
print()
print("Implication: bi-encoder recall is driven primarily by cancer type and phase match,")
print("not by biomarker-specific eligibility criteria. This is exactly what the cross-encoder fixes.")

## 2. Embedding speed on Apple Silicon (MPS)

In [ ]:
# Benchmark on 500 samples
t0 = time.time()
sample_embs = model.encode(chunks, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
elapsed = time.time() - t0

print(f"500 embeddings: {elapsed:.1f}s")
print(f"Throughput: {500/elapsed:.0f} records/sec")
print(f"Estimated for 10K: {10000/500*elapsed/60:.1f} minutes")
print(f"Estimated for 60K: {60000/500*elapsed/60:.1f} minutes")
print(f"Shape: {sample_embs.shape}")

## 3. FAISS index — build and sanity check

In [ ]:
# Load full 10K embeddings from disk (built by faiss_index.py)
INDEX_PATH = Path("../data/faiss_index.bin")
NCTIDS_PATH = Path("../data/nct_ids.npy")

if INDEX_PATH.exists():
    index = faiss.read_index(str(INDEX_PATH))
    nct_ids = np.load(str(NCTIDS_PATH), allow_pickle=True)
    print(f"Index loaded: {index.ntotal:,} vectors, dim={index.d}")
else:
    print("Index not built yet — run src/embeddings/faiss_index.py first")

## 4. Labeled query set — constructing a small eval set

In [ ]:
# 10 patient profiles with known matching NCT IDs.
# These are manually constructed from real trials in our 10K set.
# Used to compute Precision@5 and MRR for the bi-encoder baseline.

# Format: (query_text, [relevant_nct_ids])
# Populated after inspecting the DB — see cell below
EVAL_QUERIES = [
    (
        "58-year-old female with metastatic triple-negative breast cancer, BRCA2 mutation positive, "
        "2 prior chemotherapy lines, ECOG 1, no prior PARP inhibitor",
        []  # fill after DB inspection
    ),
    (
        "65-year-old male with stage IIIB non-small cell lung cancer, EGFR exon 19 deletion, "
        "no prior targeted therapy, ECOG 0",
        []
    ),
    (
        "45-year-old female with metastatic colorectal cancer, KRAS G12C mutation, "
        "MSS, 3 prior lines of therapy, liver metastases",
        []
    ),
    (
        "70-year-old male with newly diagnosed AML, FLT3-ITD positive, "
        "fit for intensive chemotherapy, ECOG 1",
        []
    ),
    (
        "52-year-old female with HR-positive HER2-negative metastatic breast cancer, "
        "CDK4/6 inhibitor refractory, PI3K mutation, ECOG 1",
        []
    ),
]

print(f"{len(EVAL_QUERIES)} evaluation queries defined")
print("NOTE: relevant NCT IDs need to be filled in after DB inspection (next cell)")

In [ ]:
# Search DB for relevant trials to label eval set
# Look for BRCA2 + breast cancer trials for query 1
brca_trials = list(db["trials"].rows_where(
    "eligibility_text LIKE ? AND conditions LIKE ?",
    ["%BRCA%", "%breast%"]
))
print(f"BRCA + breast cancer trials in DB: {len(brca_trials)}")
for t in brca_trials[:5]:
    print(f"  {t['nct_id']}: {t['brief_title'][:80]}")

In [ ]:
# Search for EGFR lung cancer trials for query 2
egfr_trials = list(db["trials"].rows_where(
    "eligibility_text LIKE ? AND conditions LIKE ?",
    ["%EGFR%", "%lung%"]
))
print(f"EGFR + lung cancer trials in DB: {len(egfr_trials)}")
for t in egfr_trials[:5]:
    print(f"  {t['nct_id']}: {t['brief_title'][:80]}")

## 5. Retrieval quality — Precision@5 and MRR baseline

In [ ]:
# Run after filling in EVAL_QUERIES with labeled NCT IDs and building the FAISS index

def precision_at_k(retrieved: list, relevant: set, k: int) -> float:
    return len(set(retrieved[:k]) & relevant) / k

def reciprocal_rank(retrieved: list, relevant: set) -> float:
    for i, nct in enumerate(retrieved):
        if nct in relevant:
            return 1.0 / (i + 1)
    return 0.0

def retrieve(query: str, index, nct_ids, model, k: int = 50) -> list:
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    _, indices = index.search(q_emb, k)
    return [nct_ids[i] for i in indices[0]]

if INDEX_PATH.exists():
    p5_scores, mrr_scores = [], []
    for query, relevant_ncts in EVAL_QUERIES:
        if not relevant_ncts:
            continue
        retrieved = retrieve(query, index, nct_ids, model, k=50)
        relevant = set(relevant_ncts)
        p5 = precision_at_k(retrieved, relevant, k=5)
        rr = reciprocal_rank(retrieved, relevant)
        p5_scores.append(p5)
        mrr_scores.append(rr)
        print(f"Query: {query[:60]}...")
        print(f"  P@5={p5:.2f}, RR={rr:.3f}")
        print(f"  Top-5: {retrieved[:5]}")
        print()

    if p5_scores:
        print(f"\nBi-encoder baseline (all-MiniLM-L6-v2):")
        print(f"  Mean Precision@5: {np.mean(p5_scores):.3f}")
        print(f"  MRR:              {np.mean(mrr_scores):.3f}")
else:
    print("Build the FAISS index first, then rerun this cell")

## 6. Where MiniLM fails — expected failure modes

Based on the truncation analysis:

**Will work OK:**
- Cancer type matching (breast cancer, lung cancer, AML) — in title and conditions, always within 256 tokens
- Phase matching (Phase 1/2/3) — short field, always visible
- Status matching

**Will fail or degrade:**
- Biomarker specificity (BRCA2, EGFR exon 19, KRAS G12C, FLT3-ITD) — often buried deep in eligibility criteria, after truncation point
- Prior treatment line requirements — almost always in eligibility body
- ECOG score cutoffs — in eligibility body
- Exclusion criteria matching — last in eligibility text, nearly always truncated

**Planned upgrade path (Component 4):**
- Switch bi-encoder to `BioLinkBERT-large` — trained on biomedical text, better biomarker term alignment
- Cross-encoder reranking operates on full text — this is where biomarker specificity gets properly scored
- Expected Precision@5 lift from reranking: to be measured in `04_retrieval_eval.ipynb`